In [ ]:
# 입력 예시 데이터
from scipy import stats
import numpy as np


#예시 값
logpress = [1, 1, 0, 1]    # 원인 4개가 각각 살았냐 죽었냐

streaming = [1, 0, 1, 0]
snapkv    = [1, 0, 0, 1]
pyramid   = [0, 1, 0, 1]
h2o       = [1, 1, 0, 0]

baseline = {"snapkv" : snapkv, "streaming" : streaming, "h2o" : h2o, "pyramid": pyramid}

# 원인 줄 메타데이터 (층 나누기용)
cause    = [2, 5, 7, 8]                  # 원인 줄 번호
is_error = {2: True, 5: False, 7: False, 8: True} # 2: 에러 맞음 5: 에러 아님 7: 에러 아님 8: 에러 맞음
pos      = {2: 0.30, 5: 0.55, 7: 0.80, 8: 0.90}

overall = [0,1,2,3]
non_error = [i for i in range(4) if not is_error[cause[i]]]
middle = [i for i in range (4) if 0.25 <= pos[cause[i]] <= 0.75]

strata = {"overall" : overall, "non_error" : non_error, "middle" : middle}


In [ ]:
def preservation(result, target): # 보존율 함수(결과, 타겟(압축기))
    return sum(1 for c in target if result[c]) / len(target) 
# target 안에 원소 c를 하나씩 꺼내서, 만약 c번째 인덱스 결과배열에서 값이 true(1)이라면 + 1 합계 구하기. / 타겟의 전체길이. 
# --> true가 전체 타겟에서 얼마나 남아있는가?의 비율.

print("-- 층별 보존율 -- ")
print(f"{'method':10}", *[f"{s:>10}" for s in strata]) #왼쪽기준으로 10칸 확보, *는 내용물 출력(리스트 언팩킹), :>10은 오른쪽 버전
for name, res in {"logpress" : logpress, **baseline}.items(): # **(딕셔너리 언팩킹)
    rates = [preservation(res, t) for t in strata.values()]
    print(f"{name:10}", *[f"{r:>10.2f}" for r in rates])

# 딕셔너리가 만들어짐 -> logpress: logpress 라는 한쌍을 만들고 그다음에 baseline에 있던 애들 언팩킹해서 하나의 딕셔너리로 만들기
# 그다음에 items()가 또 한쌍의 덩어리로 내뱉고/ name,res가 또 그 한덩어리를 쪼개서 / name, res로 만듦
# 또 그 쪼개진 res에 preservation안에 넣고 각 측마다 값들을 계산. rate 리스트에 결과 저장. 
# rate리스트에 있던 값들을 이쁘게 출력
# 루프는 name,res를 다 완료하고 각 rates를 바로 출력.


-- 층별 보존율 -- 
method        overall  non_error     middle
logpress         0.75       0.50       1.00
snapkv           0.50       0.00       0.50
streaming        0.50       0.50       0.50
h2o              0.50       0.50       1.00
pyramid          0.50       0.50       0.50


In [18]:
from math import comb

# n값은 동전 던지기 총 횟수(=의견이 엇갈린 횟수)
# k값은 동전 앞면이 나온 횟수 (=원인을 더 적게 맞힌 쪽 횟수)
# 조합으로 확률을 누적하는 과정 자체가 멕네마 검정 자체.
# 실력이 똑같다면? (0.5) 엇갈린 것이 우연인가?를 알아야함.

def binom_cdf_half(k,n): # 원인을 잘 못살린 값을 받아서 동전던지기 확률을 구함. 
    # 실력이 같으면 엇갈림은 운이라는 귀무가설에서 0.5 ** n. 즉 엇갈림이 동전던지기 확률인가를 구함.
    pmf = 0.5 ** n  # 동전을 n번 던지는데, 특정 결과 하나가 나올 확률. 그래서 0.5 n제곱 형태. 0번째 돌렸을때 확률.
    cdf = pmf # 0번(앞면 0개)일 때 확률부터 담고 시작. cdf에 첫 항 저장된 상태
    for i in range(1, k+1): # 0번~k번(앞면 0개부터 min개까지) 확률을 다 누적 = "이만큼 또는 더 극단적일 확률"
        pmf *= (n - i + 1) / i # 다음항 계산.(combiantion 점화식 자체) pmf(개별 확률질량함수)
        cdf += pmf # pmf의 누적(누적 확률질량함수)
    return cdf

def mcnemar(x,y, idx): #치우친 결과가 우연히 나올 확률? -> 누적 확률
    p = q = 0
    for i in idx:
        if x[i] != y[i]: #두쌍을 비교, 같은건 제외하고 값이 다를때만 카운트
            if x[i] == 1: # x가 1이면 p에 +1
                p += 1
            else: #아니면 q에 +1
                q += 1
    n = p+q # 얼마나 다른게 있었냐를 저장. ex) x = [1 , 0, 0, 1] y = [0, 1, 0, 1] --> n == 2
    if n == 0: return p,q,1.0 # 다른게 없었다면? (n == 0) 0,0,1.0 반환하게 됨.
    
    cdf = binom_cdf_half(min(p,q),n) # 원인을 더 잘 살리지 못한 쪽을 binom함수로 넘김. (큰값이 기준이던 작은값이던 결과는 수학적으로 동일. 계산이 편한 작은값 활용)
    p_value = min(2*cdf, 1.0) # 분포의 좌우대칭성 이용. 반쪽만 측정후, 2배. 왜?
    return p,q, p_value
# 왜 좌우대칭성이 되는가? 두 모델의 성능이 같다, 즉 비율이 같다는 가정에서 출발. 
# 확률이 정확히 0.5일때, 이항분포 그래프를 그리면 좌우대칭이 됨.(수학적 성질)
# 조합의 성질또한 대칭을 이룸. 
# 통계학에서는 이를 양측 검정이라 부르며, 검증하고자 하는 목표가 방향과 상관없이 극단적인 격차가 벌어지냐?이기 때문에 
# 구하기 쉬운 작은 값 쪽의 cdf를 구한뒤 단순 *2를 해주면 양쪽 극단의 확률을 포괄하는 p_value값이 완성됨.


#bh보정 전의 raw값들
results = []
print("BH 보정전 값들\n")

for b_name, base in baseline.items(): # 베이스라인 딕셔너리 분리해서 꺼내서
    for s_name, idx in strata.items(): # 모델 이름과, 번호로 나눠서
        p,q,p_value = mcnemar(logpress, base, idx) # 각 층별로 멕네마 검정 돌리기
        results.append([b_name, s_name, p, q, p_value]) # 결과를 results에 append
        print(f"{b_name:10} {s_name:10} p = {p} q = {q} p_value = {round(p_value,3)}") # 출력형식


BH 보정전 값들

snapkv     overall    p = 1 q = 0 p_value = 1.0
snapkv     non_error  p = 1 q = 0 p_value = 1.0
snapkv     middle     p = 1 q = 0 p_value = 1.0
streaming  overall    p = 2 q = 1 p_value = 1.0
streaming  non_error  p = 1 q = 1 p_value = 1.0
streaming  middle     p = 1 q = 0 p_value = 1.0
h2o        overall    p = 1 q = 0 p_value = 1.0
h2o        non_error  p = 0 q = 0 p_value = 1.0
h2o        middle     p = 0 q = 0 p_value = 1.0
pyramid    overall    p = 1 q = 0 p_value = 1.0
pyramid    non_error  p = 0 q = 0 p_value = 1.0
pyramid    middle     p = 1 q = 0 p_value = 1.0


In [ ]:
def bh_fdr(pvals, alpha=0.05):
    m = len(pvals) # 테스트 1번 = p-value 1개 탄생 = 리스트에 1개 추가 = 테스트 횟수가 나옴
    order = sorted(range(m), key=lambda i: pvals[i])   # p값 오름차순 정렬
    p_adj = [0.0] * m # 테스트 횟수만큼 빈리스트 만들기
    prev = 1.0
    for rank in range(m, 0, -1):        # 뒤(큰 p)에서 앞으로
        i = order[rank - 1] # 원래 값 저장.
        prev = min(prev, pvals[i] * m / rank) #BH보정 공식 
        p_adj[i] = prev # 결과 저장
    return p_adj                        # ← 보정된 p값 반환

results = []
for b_name, base in baseline.items():
    for s_name, idx in strata.items():
        b,c,p = mcnemar(logpress, base, idx)
        results.append((b_name, s_name, b,c,p)) # 멕네마 거친것을 결과에 저장
        
p_adj = bh_fdr([r[4] for r in results], alpha = 0.05) # 보정 4번째 인덱스에 있는 값을 가져옴(p_value)

for r,pa in zip(results, p_adj):
    b_name, s_name, l_win, b_win , _ = r
    if l_win > b_win: verdict = f"logpress 우세 ({l_win} vs {b_win})"
    elif b_win > l_win: verdict = f"{b_name} 우세  ({b_win} vs {l_win})"
    else:
        verdict = "동률"
    print(f"[{s_name:10}] logpress vs {b_name:10} -> {verdict}, p_bh = {pa:.3f}")
        
hard = [(r,pa) for r, pa in zip(results, p_adj) if r[1] != "overall"]
passed = all(pa < 0.05 and r[2] > r[3] for r, pa in hard)
print("\nTier-1:", "Pass" if passed else "Fail(표본이 작으면 정상)")


[overall   ] logpress vs snapkv     -> logpress 우세 (1 vs 0), p_bh = 1.000
[non_error ] logpress vs snapkv     -> logpress 우세 (1 vs 0), p_bh = 1.000
[middle    ] logpress vs snapkv     -> logpress 우세 (1 vs 0), p_bh = 1.000
[overall   ] logpress vs streaming  -> logpress 우세 (2 vs 1), p_bh = 1.000
[non_error ] logpress vs streaming  -> 동률, p_bh = 1.000
[middle    ] logpress vs streaming  -> logpress 우세 (1 vs 0), p_bh = 1.000
[overall   ] logpress vs h2o        -> logpress 우세 (1 vs 0), p_bh = 1.000
[non_error ] logpress vs h2o        -> 동률, p_bh = 1.000
[middle    ] logpress vs h2o        -> 동률, p_bh = 1.000
[overall   ] logpress vs pyramid    -> logpress 우세 (1 vs 0), p_bh = 1.000
[non_error ] logpress vs pyramid    -> 동률, p_bh = 1.000
[middle    ] logpress vs pyramid    -> logpress 우세 (1 vs 0), p_bh = 1.000

Tier-1: Fail(표본이 작으면 정상)


>BY: 얽힘이 '제멋대로'일 때의 보험.
>우리 60검정은 전부 LogPress를 공유해 '같은 방향' 얽힘일 가능성이 높음(증명은 아님)
>→ BH를 주 판정으로 쓰되, 확신 못 하므로 Holm을 병기해 방어